# 06 — Agentic Loop：Agent 主循环

这是整门课最核心的章节。前五章分别实现了：

| 章节 | 模块 | 作用 |
|------|------|------|
| 01 | `LLMClient` | 封装 Ollama API，支持流式输出和工具调用 |
| 02 | `ContextManager` | 管理对话历史，估算 token 用量 |
| 03 | `ToolRegistry` / `Tool` | 工具注册与调度框架 |
| 04 | 文件工具 | 读写文件、列目录、glob 搜索 |
| 05 | 网络工具 | Web 搜索、URL 抓取 |

本章把这些模块串起来，实现 **Agent 自主循环（Agentic Loop）**：LLM 可以自主决定调用哪些工具、调用多少轮，直到完成任务。

## 1. 核心概念：Agent 主循环

Agent 不是一次性问答，而是一个**循环**：

```
用户输入
   │
   ▼
┌─────────────────────────────────────────────┐
│  第 1 轮                                     │
│  LLM 决定：调用 list_directory               │
│  执行工具 → 返回目录列表                      │
│  把结果追加到上下文                           │
└─────────────────────────────────────────────┘
   │
   ▼
┌─────────────────────────────────────────────┐
│  第 2 轮                                     │
│  LLM 决定：调用 read_file 读某个文件          │
│  执行工具 → 返回文件内容                      │
│  把结果追加到上下文                           │
└─────────────────────────────────────────────┘
   │
   ▼
┌─────────────────────────────────────────────┐
│  第 3 轮                                     │
│  LLM 不调用工具，直接生成最终回答              │
│  finish_reason == "stop" → 退出循环           │
└─────────────────────────────────────────────┘
   │
   ▼
返回给用户
```

**两个退出条件**：
1. LLM 不再调用工具（`finish_reason == "stop"`）
2. 达到最大轮次 `max_turns`（防止无限循环）

**事件驱动设计**：循环用异步生成器实现，每一步都 `yield` 一个 `AgentEvent`，外层代码可以实时响应（打印、记录、展示 UI 等）。

## 2. AgentEvent：事件类型定义

用枚举列举所有可能的事件，让外层代码用 `match` 或 `if` 精确处理每种情况。

In [ ]:
import sys
sys.path.insert(0, "..")

from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Any


class AgentEventType(Enum):
    """Agent 主循环的所有事件类型"""
    # 整个 Agent 生命周期
    AGENT_START = auto()        # Agent 开始执行
    AGENT_END = auto()          # Agent 正常结束
    AGENT_ERROR = auto()        # Agent 遇到不可恢复的错误

    # 文本流
    TEXT_DELTA = auto()         # 流式文本片段（增量）
    TEXT_COMPLETE = auto()      # 本轮文本已完整（完整文本）

    # 工具调用
    TOOL_CALL_START = auto()    # 即将执行某个工具
    TOOL_CALL_COMPLETE = auto() # 工具执行完成（含结果）

    # 轮次
    ROUND_START = auto()        # 新一轮 LLM 调用开始
    ROUND_END = auto()          # 本轮结束（无论是否调用了工具）


@dataclass
class AgentEvent:
    """Agent 事件，携带类型和可选数据"""
    type: AgentEventType
    data: Any = None            # 事件携带的数据，结构由 type 决定

    def __repr__(self) -> str:
        data_repr = repr(self.data)[:80] if self.data is not None else "None"
        return f"AgentEvent({self.type.name}, {data_repr})"


# 快速验证
print(AgentEvent(AgentEventType.AGENT_START))
print(AgentEvent(AgentEventType.TEXT_DELTA, data="Hello"))
print(AgentEvent(AgentEventType.TOOL_CALL_START, data={"name": "read_file", "params": {"path": "foo.py"}}))
print("所有事件类型：", [e.name for e in AgentEventType])

## 3. 流式 Tool Call 解析

这是整个实现里最容易出 bug 的地方，单独讲清楚。

### 问题：流式响应中 tool_calls 是分片发来的

非流式响应中，`tool_calls` 是一个完整的列表：
```json
{"tool_calls": [{"id": "call_abc", "function": {"name": "read_file", "arguments": "{\"path\":\"foo.py\"}"}}]}
```

流式响应中，同样的内容被拆成多个 chunk：
```
chunk 1: delta.tool_calls = [{index:0, id:"call_abc", function:{name:"read_file", arguments:""}}]
chunk 2: delta.tool_calls = [{index:0, function:{arguments:"{\\"path\\"}}"}}]
chunk 3: delta.tool_calls = [{index:0, function:{arguments:":\\"foo.py\\"}"}}]
```

需要按 `index` 把每个 chunk 的增量**拼接**起来，得到完整的 tool_call。

### 拼接逻辑

In [ ]:
import json
from typing import Any


def accumulate_tool_call_chunk(
    buffer: dict[int, dict],
    chunk_tool_calls: list,
) -> None:
    """
    把一个流式 chunk 里的 tool_calls 增量合并进 buffer。

    buffer 结构：
        {index: {"id": str, "name": str, "arguments": str}}

    注意：
    - id 和 name 只在第一个 chunk 里出现，后续 chunk 不重复
    - arguments 是字符串，每个 chunk 追加（不是覆盖）
    """
    for tc in chunk_tool_calls:
        idx = tc.index  # 区分同一次调用中多个并行工具的序号

        if idx not in buffer:
            # 第一次见到这个 index，初始化槽位
            buffer[idx] = {"id": "", "name": "", "arguments": ""}

        slot = buffer[idx]

        # id 只在首 chunk 出现
        if tc.id:
            slot["id"] = tc.id

        # function.name 只在首 chunk 出现
        if tc.function and tc.function.name:
            slot["name"] = tc.function.name

        # function.arguments 是增量字符串，需要追加
        if tc.function and tc.function.arguments:
            slot["arguments"] += tc.function.arguments


def finalize_tool_calls(buffer: dict[int, dict]) -> list[dict]:
    """
    把 buffer 里所有拼接完成的 tool_call 转为标准格式。
    同时解析 arguments JSON 字符串为 dict。
    """
    result = []
    for idx in sorted(buffer.keys()):
        slot = buffer[idx]
        try:
            params = json.loads(slot["arguments"]) if slot["arguments"] else {}
        except json.JSONDecodeError as e:
            # arguments 格式错误：保留原始字符串，让调用层处理
            params = {"_raw_arguments": slot["arguments"], "_parse_error": str(e)}
        result.append({
            "id": slot["id"],
            "name": slot["name"],
            "params": params,
        })
    return result


# 用模拟数据验证拼接逻辑
class FakeFunction:
    def __init__(self, name="", arguments=""):
        self.name = name
        self.arguments = arguments

class FakeTC:
    def __init__(self, index, id="", name="", arguments=""):
        self.index = index
        self.id = id
        self.function = FakeFunction(name, arguments)

buffer: dict[int, dict] = {}

# 模拟三个 chunk 到达
accumulate_tool_call_chunk(buffer, [
    FakeTC(index=0, id="call_abc", name="read_file", arguments="")
])
accumulate_tool_call_chunk(buffer, [
    FakeTC(index=0, arguments='{"path":"')
])
accumulate_tool_call_chunk(buffer, [
    FakeTC(index=0, arguments='foo.py"}')
])

print("拼接后 buffer：", buffer)
print("最终 tool_calls：", finalize_tool_calls(buffer))

## 4. AgenticLoop 主函数

现在实现核心的异步生成器。每一步都 `yield AgentEvent`，外层实时消费。

In [ ]:
import asyncio
from typing import AsyncGenerator

# 导入前五章的模块（需要 src/ 在 sys.path 里）
try:
    from src.llm_client import LLMClient
    from src.context_manager import ContextManager
    from src.tool_framework import ToolRegistry, ToolResult
except ImportError as e:
    print(f"导入失败（确认 src/ 存在并包含前五章代码）：{e}")
    raise


async def agentic_loop(
    user_message: str,
    llm: LLMClient,
    ctx: ContextManager,
    registry: ToolRegistry,
    max_turns: int = 10,
) -> AsyncGenerator[AgentEvent, None]:
    """
    Agent 主循环（异步生成器）。

    每完成一步就 yield 一个 AgentEvent，外层可以实时响应。
    循环在以下两种情况退出：
      1. LLM 不再调用工具（finish_reason == "stop"）
      2. 达到 max_turns 上限

    参数：
      user_message : 用户输入的问题或指令
      llm          : LLMClient 实例
      ctx          : ContextManager 实例（含历史上下文）
      registry     : 注册了可用工具的 ToolRegistry
      max_turns    : 最大循环轮次，防止无限循环
    """
    # 统计信息
    total_rounds = 0
    total_tool_calls = 0

    yield AgentEvent(AgentEventType.AGENT_START, data={"message": user_message})

    # 把用户消息加入上下文
    ctx.add_user_message(user_message)

    try:
        for turn in range(max_turns):
            total_rounds += 1
            yield AgentEvent(AgentEventType.ROUND_START, data={"round": turn + 1})

            # ── 调用 LLM，流式获取响应 ──────────────────────────────────
            tools = registry.get_schemas() if registry.list_tools() else None
            stream = await llm.chat_completion(
                ctx.get_messages(),
                stream=True,
                tools=tools,
            )

            # ── 流式读取，同时收集文本和 tool_calls ───────────────────────
            accumulated_text = ""
            tool_calls_buffer: dict[int, dict] = {}  # index -> slot
            finish_reason = None

            async for chunk in stream:
                # chunk 是 openai.types.chat.ChatCompletionChunk
                choice = chunk.choices[0] if chunk.choices else None
                if choice is None:
                    continue

                delta = choice.delta
                finish_reason = choice.finish_reason or finish_reason

                # 文本增量
                if delta.content:
                    accumulated_text += delta.content
                    yield AgentEvent(AgentEventType.TEXT_DELTA, data=delta.content)

                # tool_calls 增量（按 index 拼接）
                if delta.tool_calls:
                    accumulate_tool_call_chunk(tool_calls_buffer, delta.tool_calls)

            # ── 流式结束，处理收集到的内容 ────────────────────────────────

            # 把完整文本告知外层
            if accumulated_text:
                yield AgentEvent(AgentEventType.TEXT_COMPLETE, data=accumulated_text)

            # 解析完整 tool_calls
            final_tool_calls = finalize_tool_calls(tool_calls_buffer)

            if final_tool_calls:
                # ── 有工具调用：执行工具，结果追回上下文，继续下一轮 ────────

                # 把 assistant 消息（含 tool_calls）加入上下文
                # 需要转成 OpenAI 格式的 tool_calls 列表
                openai_tool_calls = [
                    {
                        "id": tc["id"],
                        "type": "function",
                        "function": {
                            "name": tc["name"],
                            "arguments": json.dumps(tc["params"])
                                if not isinstance(tc["params"], str)
                                else tc["params"],
                        },
                    }
                    for tc in final_tool_calls
                ]
                ctx.add_assistant_message(accumulated_text, tool_calls=openai_tool_calls)

                # 逐个执行工具
                for tc in final_tool_calls:
                    tool_name = tc["name"]
                    tool_params = tc["params"]
                    call_id = tc["id"]

                    yield AgentEvent(
                        AgentEventType.TOOL_CALL_START,
                        data={"name": tool_name, "params": tool_params, "call_id": call_id},
                    )

                    # 执行工具（捕获异常，不让单个工具失败崩溃整个循环）
                    try:
                        result: ToolResult = await registry.invoke(tool_name, tool_params)
                    except Exception as exc:
                        result = ToolResult.error(f"工具执行异常：{exc}")

                    total_tool_calls += 1

                    # 把工具结果加入上下文
                    ctx.add_tool_result(call_id, result.content)

                    yield AgentEvent(
                        AgentEventType.TOOL_CALL_COMPLETE,
                        data={
                            "name": tool_name,
                            "call_id": call_id,
                            "success": result.success,
                            "content": result.content,
                            "is_error": result.is_error,
                        },
                    )

                yield AgentEvent(AgentEventType.ROUND_END, data={"round": turn + 1})
                # 继续下一轮

            else:
                # ── 无工具调用：LLM 已生成最终答案，退出循环 ────────────────
                if accumulated_text:
                    ctx.add_assistant_message(accumulated_text)
                yield AgentEvent(AgentEventType.ROUND_END, data={"round": turn + 1})
                break

        else:
            # for 循环正常跑完（达到 max_turns 还没 stop）
            yield AgentEvent(
                AgentEventType.AGENT_ERROR,
                data={"error": f"达到最大轮次 {max_turns}，强制退出"},
            )

    except Exception as exc:
        yield AgentEvent(AgentEventType.AGENT_ERROR, data={"error": str(exc)})
        raise

    finally:
        yield AgentEvent(
            AgentEventType.AGENT_END,
            data={
                "total_rounds": total_rounds,
                "total_tool_calls": total_tool_calls,
                "context_tokens": ctx.estimated_tokens,
            },
        )


print("agentic_loop 定义完成")

## 5. 实时打印事件：print_events()

一个简单的消费函数，把事件流实时打印到终端。

In [ ]:
import time


async def print_events(
    event_gen: AsyncGenerator[AgentEvent, None],
    show_tool_content: bool = True,
    max_content_chars: int = 400,
) -> None:
    """
    消费 AgentEvent 异步生成器，实时打印到终端。

    参数：
      event_gen         : agentic_loop 返回的生成器
      show_tool_content : 是否打印工具返回内容（内容很长时可关掉）
      max_content_chars : 工具内容截断长度
    """
    start_time = time.time()

    async for event in event_gen:
        match event.type:

            case AgentEventType.AGENT_START:
                print(f"\n{'='*60}")
                print(f"[Agent] 开始执行")
                print(f"[Agent] 用户消息：{event.data['message'][:100]}")
                print(f"{'='*60}")

            case AgentEventType.ROUND_START:
                print(f"\n--- 第 {event.data['round']} 轮 ---")

            case AgentEventType.TEXT_DELTA:
                # 流式文本：不换行，直接追加输出
                print(event.data, end="", flush=True)

            case AgentEventType.TEXT_COMPLETE:
                # 流式文本结束，换行
                print()  # 换行

            case AgentEventType.TOOL_CALL_START:
                d = event.data
                print(f"\n[工具] 调用 {d['name']}")
                params_str = json.dumps(d["params"], ensure_ascii=False)
                if len(params_str) > 200:
                    params_str = params_str[:200] + "..."
                print(f"       参数：{params_str}")

            case AgentEventType.TOOL_CALL_COMPLETE:
                d = event.data
                status = "成功" if d["success"] else "失败"
                print(f"[工具] {d['name']} → {status}")
                if show_tool_content:
                    content = d["content"]
                    if len(content) > max_content_chars:
                        content = content[:max_content_chars] + f"...（共 {len(d['content'])} 字符）"
                    print(f"       返回：{content}")

            case AgentEventType.ROUND_END:
                pass  # 轮次结束无需额外输出

            case AgentEventType.AGENT_END:
                elapsed = time.time() - start_time
                d = event.data
                print(f"\n{'='*60}")
                print(f"[Agent] 完成")
                print(f"        总轮次：{d['total_rounds']}")
                print(f"        工具调用次数：{d['total_tool_calls']}")
                print(f"        上下文 token 估算：{d['context_tokens']}")
                print(f"        耗时：{elapsed:.1f}s")
                print(f"{'='*60}")

            case AgentEventType.AGENT_ERROR:
                print(f"\n[错误] {event.data['error']}")


print("print_events 定义完成")

## 6. 注册工具并组装 Agent

把第 04 章的文件工具注册进来，构建一个完整的 Agent 实例。

In [ ]:
import os

try:
    from src.llm_client import LLMClient
    from src.context_manager import ContextManager, build_system_prompt
    from src.tool_framework import ToolRegistry
    from src.file_tools import ReadFileTool, ListDirectoryTool, GlobTool
except ImportError as e:
    print(f"[警告] 导入失败，请确认前五章代码已保存到 src/：{e}")
    raise

# 工作目录：course/ 的上一级（项目根目录）
WORKING_DIR = os.path.abspath("..")

def build_agent(
    system_prompt: str = "",
    working_dir: str = WORKING_DIR,
):
    """构建并返回 (llm, ctx, registry) 三元组"""
    llm = LLMClient(
        model="gpt-oss:120b",
        base_url="http://localhost:11434/v1",
        api_key="ollama",
        temperature=0.3,
    )

    ctx = ContextManager(
        system_prompt=system_prompt or build_system_prompt(working_dir)
    )

    registry = ToolRegistry()
    registry.register(ListDirectoryTool(cwd=working_dir))
    registry.register(ReadFileTool(cwd=working_dir))
    registry.register(GlobTool(cwd=working_dir))

    return llm, ctx, registry


llm, ctx, registry = build_agent()
print("已注册工具：", [t.name for t in registry.list_tools()])
print("工作目录：", WORKING_DIR)

## 7. 完整测试：给 Agent 一个多步骤任务

任务：「列出项目根目录，然后读 course/01_ollama_llm_client.ipynb 的前 20 行」

Agent 需要：
1. 第 1 轮：调用 `list_directory` 查看根目录
2. 第 2 轮：调用 `read_file` 读取指定文件
3. 第 3 轮：生成最终回答，退出

运行前请确认 Ollama 已启动（`ollama serve`）。

In [ ]:
async def test_full_loop():
    """完整的 Agent 循环测试"""
    llm, ctx, registry = build_agent()

    task = (
        "请完成两件事："
        "1. 列出项目根目录（即工作目录）下的所有文件和文件夹；"
        "2. 读取 course/01_ollama_llm_client.ipynb 文件的前 30 行内容。"
        "最后用中文总结你看到了什么。"
    )

    try:
        gen = agentic_loop(
            user_message=task,
            llm=llm,
            ctx=ctx,
            registry=registry,
            max_turns=8,
        )
        await print_events(gen, show_tool_content=True, max_content_chars=300)
    finally:
        await llm.close()


await test_full_loop()

## 8. 错误场景测试

验证 Agent 在遇到以下错误时能正确处理并继续运行：
- 调用不存在的工具
- 读取不存在的文件
- 工具参数缺失

In [ ]:
async def test_error_handling():
    """错误场景：Agent 试图读取一个不存在的文件"""
    llm, ctx, registry = build_agent()

    task = "请读取文件 /tmp/this_file_does_not_exist_xyz.txt 的内容。"

    print("[测试] 错误场景：读取不存在的文件")
    print("期望行为：工具返回错误，LLM 根据错误信息回答用户")

    collected_events = []

    try:
        gen = agentic_loop(
            user_message=task,
            llm=llm,
            ctx=ctx,
            registry=registry,
            max_turns=4,
        )

        async for event in gen:
            collected_events.append(event)
            # 只打印关键事件
            if event.type == AgentEventType.TOOL_CALL_COMPLETE:
                d = event.data
                print(f"[工具] {d['name']} → {'成功' if d['success'] else '失败'}")
                print(f"       内容：{d['content'][:200]}")
            elif event.type == AgentEventType.TEXT_COMPLETE:
                print(f"[LLM 回答] {event.data[:300]}")
            elif event.type == AgentEventType.AGENT_END:
                print(f"[Agent 结束] 总轮次={event.data['total_rounds']}，工具调用={event.data['total_tool_calls']}")
    finally:
        await llm.close()

    # 检查是否收到了 TOOL_CALL_COMPLETE 事件（即工具被调用了）
    tool_complete_events = [e for e in collected_events if e.type == AgentEventType.TOOL_CALL_COMPLETE]
    if tool_complete_events:
        tc = tool_complete_events[0]
        print(f"\n[验证] 工具调用成功捕获，is_error={tc.data['is_error']}")
    else:
        print("\n[验证] 未检测到工具调用事件（可能 LLM 直接拒绝了请求）")


await test_error_handling()

## 9. max_turns 上限测试

验证当 `max_turns` 很小时，Agent 会在 `AGENT_ERROR` 事件里报告强制退出，而不是静默挂死。

In [ ]:
async def test_max_turns():
    """max_turns 上限：强制 Agent 在 2 轮内结束"""
    llm, ctx, registry = build_agent()

    # 给一个需要多步骤的任务，但 max_turns=2
    task = (
        "请依次完成："
        "1. 列出根目录；"
        "2. 列出 course/ 目录；"
        "3. 读取 course/01_ollama_llm_client.ipynb 前 10 行；"
        "4. 总结内容。"
    )

    print("[测试] max_turns=2，期望在第 2 轮后触发 AGENT_ERROR")

    try:
        gen = agentic_loop(
            user_message=task,
            llm=llm,
            ctx=ctx,
            registry=registry,
            max_turns=2,  # 故意设得很小
        )

        async for event in gen:
            if event.type == AgentEventType.ROUND_START:
                print(f"[轮次开始] 第 {event.data['round']} 轮")
            elif event.type == AgentEventType.TOOL_CALL_START:
                print(f"[工具] 调用 {event.data['name']}")
            elif event.type == AgentEventType.AGENT_ERROR:
                print(f"[AGENT_ERROR] {event.data['error']}")
            elif event.type == AgentEventType.AGENT_END:
                print(f"[Agent 结束] rounds={event.data['total_rounds']}")
    finally:
        await llm.close()


await test_max_turns()

## 10. 收集所有事件（用于程序化处理）

`print_events` 适合调试，生产场景更常见的需求是把事件收集成列表，做后处理。

In [ ]:
async def collect_events(
    event_gen: AsyncGenerator[AgentEvent, None],
) -> list[AgentEvent]:
    """把 AgentEvent 生成器收集为列表，用于测试和后处理"""
    events = []
    async for event in event_gen:
        events.append(event)
    return events


async def test_collect_and_inspect():
    """演示如何收集事件并做程序化分析"""
    llm, ctx, registry = build_agent()

    task = "列出项目根目录的内容"

    try:
        events = await collect_events(
            agentic_loop(task, llm, ctx, registry, max_turns=5)
        )
    finally:
        await llm.close()

    # 统计各类型事件
    from collections import Counter
    counts = Counter(e.type.name for e in events)
    print("事件统计：")
    for name, count in sorted(counts.items()):
        print(f"  {name}: {count}")

    # 提取最终回答
    text_complete = [e for e in events if e.type == AgentEventType.TEXT_COMPLETE]
    if text_complete:
        print(f"\n最终回答（最后一段）：")
        print(text_complete[-1].data[:500])

    # 提取工具调用记录
    tool_calls = [e for e in events if e.type == AgentEventType.TOOL_CALL_COMPLETE]
    print(f"\n工具调用记录 ({len(tool_calls)} 次)：")
    for tc in tool_calls:
        d = tc.data
        print(f"  {d['name']} → {'成功' if d['success'] else '失败'}")


await test_collect_and_inspect()

## 11. 保存到 src/agentic_loop.py

In [ ]:
import os

SOURCE_CODE = '''
import json
from enum import Enum, auto
from dataclasses import dataclass
from typing import Any, AsyncGenerator


class AgentEventType(Enum):
    """Agent 主循环的所有事件类型"""
    AGENT_START = auto()
    AGENT_END = auto()
    AGENT_ERROR = auto()
    TEXT_DELTA = auto()
    TEXT_COMPLETE = auto()
    TOOL_CALL_START = auto()
    TOOL_CALL_COMPLETE = auto()
    ROUND_START = auto()
    ROUND_END = auto()


@dataclass
class AgentEvent:
    """Agent 事件，携带类型和可选数据"""
    type: AgentEventType
    data: Any = None

    def __repr__(self) -> str:
        data_repr = repr(self.data)[:80] if self.data is not None else "None"
        return f"AgentEvent({self.type.name}, {data_repr})"


def accumulate_tool_call_chunk(
    buffer: dict[int, dict],
    chunk_tool_calls: list,
) -> None:
    """
    把一个流式 chunk 里的 tool_calls 增量合并进 buffer。
    buffer 结构：{index: {"id": str, "name": str, "arguments": str}}
    """
    for tc in chunk_tool_calls:
        idx = tc.index
        if idx not in buffer:
            buffer[idx] = {"id": "", "name": "", "arguments": ""}
        slot = buffer[idx]
        if tc.id:
            slot["id"] = tc.id
        if tc.function and tc.function.name:
            slot["name"] = tc.function.name
        if tc.function and tc.function.arguments:
            slot["arguments"] += tc.function.arguments


def finalize_tool_calls(buffer: dict[int, dict]) -> list[dict]:
    """
    把 buffer 里拼接完成的 tool_calls 转为标准格式，解析 arguments JSON。
    """
    result = []
    for idx in sorted(buffer.keys()):
        slot = buffer[idx]
        try:
            params = json.loads(slot["arguments"]) if slot["arguments"] else {}
        except json.JSONDecodeError as e:
            params = {"_raw_arguments": slot["arguments"], "_parse_error": str(e)}
        result.append({"id": slot["id"], "name": slot["name"], "params": params})
    return result


async def agentic_loop(
    user_message: str,
    llm,
    ctx,
    registry,
    max_turns: int = 10,
) -> AsyncGenerator[AgentEvent, None]:
    """
    Agent 主循环（异步生成器）。

    每完成一步就 yield 一个 AgentEvent。
    循环退出条件：
      1. LLM 不再调用工具（finish_reason == stop）
      2. 达到 max_turns 上限
    """
    total_rounds = 0
    total_tool_calls = 0

    yield AgentEvent(AgentEventType.AGENT_START, data={"message": user_message})
    ctx.add_user_message(user_message)

    try:
        for turn in range(max_turns):
            total_rounds += 1
            yield AgentEvent(AgentEventType.ROUND_START, data={"round": turn + 1})

            tools = registry.get_schemas() if registry.list_tools() else None
            stream = await llm.chat_completion(
                ctx.get_messages(),
                stream=True,
                tools=tools,
            )

            accumulated_text = ""
            tool_calls_buffer: dict[int, dict] = {}
            finish_reason = None

            async for chunk in stream:
                choice = chunk.choices[0] if chunk.choices else None
                if choice is None:
                    continue
                delta = choice.delta
                finish_reason = choice.finish_reason or finish_reason

                if delta.content:
                    accumulated_text += delta.content
                    yield AgentEvent(AgentEventType.TEXT_DELTA, data=delta.content)

                if delta.tool_calls:
                    accumulate_tool_call_chunk(tool_calls_buffer, delta.tool_calls)

            if accumulated_text:
                yield AgentEvent(AgentEventType.TEXT_COMPLETE, data=accumulated_text)

            final_tool_calls = finalize_tool_calls(tool_calls_buffer)

            if final_tool_calls:
                openai_tool_calls = [
                    {
                        "id": tc["id"],
                        "type": "function",
                        "function": {
                            "name": tc["name"],
                            "arguments": json.dumps(tc["params"])
                                if not isinstance(tc["params"], str)
                                else tc["params"],
                        },
                    }
                    for tc in final_tool_calls
                ]
                ctx.add_assistant_message(accumulated_text, tool_calls=openai_tool_calls)

                for tc in final_tool_calls:
                    tool_name = tc["name"]
                    tool_params = tc["params"]
                    call_id = tc["id"]

                    yield AgentEvent(
                        AgentEventType.TOOL_CALL_START,
                        data={"name": tool_name, "params": tool_params, "call_id": call_id},
                    )

                    try:
                        result = await registry.invoke(tool_name, tool_params)
                    except Exception as exc:
                        from src.tool_framework import ToolResult
                        result = ToolResult.error(f"工具执行异常：{exc}")

                    total_tool_calls += 1
                    ctx.add_tool_result(call_id, result.content)

                    yield AgentEvent(
                        AgentEventType.TOOL_CALL_COMPLETE,
                        data={
                            "name": tool_name,
                            "call_id": call_id,
                            "success": result.success,
                            "content": result.content,
                            "is_error": result.is_error,
                        },
                    )

                yield AgentEvent(AgentEventType.ROUND_END, data={"round": turn + 1})

            else:
                if accumulated_text:
                    ctx.add_assistant_message(accumulated_text)
                yield AgentEvent(AgentEventType.ROUND_END, data={"round": turn + 1})
                break

        else:
            yield AgentEvent(
                AgentEventType.AGENT_ERROR,
                data={"error": f"达到最大轮次 {max_turns}，强制退出"},
            )

    except Exception as exc:
        yield AgentEvent(AgentEventType.AGENT_ERROR, data={"error": str(exc)})
        raise

    finally:
        yield AgentEvent(
            AgentEventType.AGENT_END,
            data={
                "total_rounds": total_rounds,
                "total_tool_calls": total_tool_calls,
                "context_tokens": ctx.estimated_tokens,
            },
        )
'''

# 写入 src/agentic_loop.py
src_dir = os.path.abspath("../src")
os.makedirs(src_dir, exist_ok=True)
output_path = os.path.join(src_dir, "agentic_loop.py")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(SOURCE_CODE.lstrip())

print(f"已保存：{output_path}")
print(f"文件大小：{os.path.getsize(output_path)} 字节")

## 小结

| 模块 | 作用 |
|------|------|
| `AgentEventType` | 枚举所有事件类型，让事件处理逻辑清晰 |
| `AgentEvent` | 统一的事件载体，携带 type 和 data |
| `accumulate_tool_call_chunk` | 把流式 tool_calls 增量按 index 拼接成完整调用 |
| `finalize_tool_calls` | 把拼接完的 buffer 转为标准格式，解析 arguments JSON |
| `agentic_loop` | 异步生成器，驱动 LLM → 工具 → LLM 的多轮循环 |
| `print_events` | 实时打印事件流，适合调试和终端交互 |
| `collect_events` | 把事件流收集为列表，适合程序化处理和测试 |

**关键设计决策**：
- 用异步生成器而非回调，外层代码用 `async for` 消费，既能实时响应也能暂停
- `AGENT_END` 在 `finally` 里 yield，确保无论正常退出还是异常都会触发
- 单个工具执行失败不崩溃整个循环，错误通过 `ToolResult.error` 传回 LLM

下一章（07）将在此基础上实现**上下文压缩**：当对话历史超过模型窗口时，自动摘要旧消息以腾出空间。